In [ ]:
import sys
sys.path.append(r'C:\Users\sadiq\Desktop\U_Vienna\codes\icm-zebrafish')
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
from ica_utils import *
from scipy.stats import zscore

In [ ]:
%reload_ext autoreload
%autoreload 2
from ica_utils import *

In [ ]:
traces = np.load('fluo_220127_F4_run2.npy')
tail_angle= np.load('220127_F4_run2_tail_angle.npy')
ic_labels = np.load('ic_labels.npy')
all_background = np.load('all_background.npy', allow_pickle=True)
all_positions = np.load('all_positions.npy', allow_pickle=True)
f_s = 5.3
n = traces.shape[0]

In [ ]:
all_positions.shape

In [ ]:
def xyz_maker(topography):
    x_val,y_val,z_val = [],[],[]
    for i in range(topography.shape[0]):
        x_val.append([topography[i,2],topography[i,2]])
        y_val.append([topography[i,2],topography[i,2]])
        z_val.append([topography[i,2],topography[i,2]])
    return x_val,y_val,z_val

from mpl_toolkits.mplot3d import axes3d
def plott3D(inferred,centers):
    x_val,y_val,z_val = xyz_maker(inferred3,centers)
    fig = plt.figure(figsize = (30,15))
    ax = fig.add_subplot(111,projection="3d")
    ax.scatter(centers[:,0],centers[:,1],centers[:,2],color='red',s=30,marker='*')
    for a in range(len(x_val)):
        ax.plot(x_val[a],y_val[a],z_val[a],lw=0.5,alpha=.7)

In [ ]:
ic_labels

In [ ]:
len(tail_angle)

In [ ]:
plt.figure(figsize=(20,2))
plt.plot(tail_angle)

In [ ]:
# resampling behavior to same number of samples as the traces

from scipy.signal import resample
aa = resample(np.abs(tail_angle),3176,domain='time')
plt.figure(figsize=(20,2))
plt.plot(aa)

In [ ]:
aa = aa[np.newaxis,:]

In [ ]:
ic_comps,IC_ft,A,mean = ica_dec(traces,n,t=0.0001,max_=500)

In [ ]:
IC_ft.shape

In [ ]:
def plott_ics(ics):        
    fig,ax = plt.subplots(ics.shape[1],1,figsize=(15,1.2*ics.shape[1]))
    for i in range(ics.shape[1]):
        ax[i].vlines(x=[562,675,1103,1963,2162,2295,2296,2297,472,473,478,447,697,1020,2581,2600],ymin=ics.T[i,:].min(),ymax=ics.T[i,:].max(),ls='--',color='g',lw = .7)
        ax[i].plot((ics.T[i,:]),label='{}'.format(i),lw=.7)
        ax[i].legend()
        
def plottings_spectrals3(n_clus,alll):
    fig,ax=plt.subplots(1,n_clus,figsize=(15,3))
    for i in range(n_clus):
        group = IC_ft[np.where(alll[:,3] == i)]
        for j in range(group.shape[0]):
            ax[i].plot(np.fft.fftshift(np.linspace(-f_s/2, f_s/2,3176)),group[j,:])
        ax[i].plot(np.fft.fftshift(np.linspace(-f_s/2, f_s/2,3176)),group.mean(0),color='black',lw=1,label='mean')
        ax[i].set_xlim([0,1])
        ax[i].set_ylim([0,5])
        ax[i].set_title('cl {} with {}'.format(i,group.shape[0]))
        ax[i].legend()

def plottings_logSpectral3(n_clus,alll):
    plt.figure(figsize=(15,8))
    for i in range(n_clus):
        group = IC_ft[np.where(alll[:,3] == i)]
        plt.plot(np.fft.fftshift(np.linspace(-f_s/2, f_s/2,3176)),np.log(group.mean(0)),lw=1,label='{}'.format(i))
        plt.xlim([0,.5])
        plt.ylim([0,2.5])
        plt.legend()

In [ ]:
# np.save('ic_comps_run2.npy',ic_comps)

In [ ]:
plott_ics(ic_comps)

In [ ]:
plt.imshow(np.corrcoef(ic_comps.T))

In [ ]:
plot_FT_spectrals(ic_comps,f_s,n)

In [ ]:
plt.plot(np.fft.fftshift(np.linspace(-f_s/2, f_s/2, ic_comps.shape[0])),IC_ft.T)
plt.xlim(0,3)

### Clustering

In [ ]:
new_mat, predictions= cluster(ic_comps,2,f_s)

In [ ]:
np.sum(predictions==0)

In [ ]:
al = np.c_[new_mat,predictions,np.arange(n)]
al.round(1)

In [ ]:
%matplotlib notebook
plot_clusters(new_mat,predictions)   # 0 are noise, 1 are signals

In [ ]:
group0 = IC_ft[np.where(al[:,3] == 0)]
group0.shape

In [ ]:
plottings_spectrals(IC_ft,2,al,f_s)

In [ ]:
plottings_logSpectral(IC_ft,2,al,f_s)

In [ ]:
# indexes to set to zeros
idx_IC_Clust_pred = np.where(predictions==0)

In [ ]:
new_ic_comps= ic_comps.copy()
new_ic_comps[:,idx_IC_Clust_pred] = 0

In [ ]:
# cleaning F1_T1 
cleaned_traces = np.dot(new_ic_comps, A.T) + mean

In [ ]:
cleaned_traces.shape

In [ ]:
%matplotlib inline
fig,ax = plt.subplots(cleaned_traces.shape[1],1,figsize=(15,1.2*cleaned_traces.shape[1]))
for i in range(cleaned_traces.shape[1]):
    ax[i].plot(traces[i,:],label=f'raw{i}',color='red',lw=.75)
    ax[i].plot(cleaned_traces[:,i]+3,label=f'clean{i}',color='blue',alpha=0.85,lw=.75)
    ax[i].vlines(x=[562,675,1103,1963,2162,2295,2296,2297,472,473,478,447,697,1020,2581,2600],ymin=traces[i,:].min(),ymax=traces[i,:].max(),alpha=.5,lw=.6,ls='--',color='g')
    ax[i].legend()

In [ ]:
# zoomed in around artifact points
%matplotlib inline
fig,ax = plt.subplots(cleaned_traces.shape[1],1,figsize=(15,1.2*cleaned_traces.shape[1]))
for i in range(cleaned_traces.shape[1]):
    ax[i].plot(traces[i,:],label='original',color='red')
    ax[i].plot(cleaned_traces[:,i]+3,label='cleaned',color='blue',alpha=0.85)
    ax[i].vlines(x=[562,675,1103,1963,2162,2295,2296,2297,472,473,478,447,697,1020,2581,2600],ymin=traces[i,:].min(),ymax=traces[i,:].max(),alpha=.5,lw=.6,ls='--',color='g')
    ax[i].set_xlim([550,2650])
    ax[i].legend()

#### Seven clusters

In [ ]:
new_mat7, predictions7= cluster(ic_comps,7,f_s)

In [ ]:
%matplotlib notebook
plot_clusters(new_mat7,predictions7)

In [ ]:
al7 = np.c_[new_mat7.round(2),predictions7.round(1),np.arange(n)]
al7

In [ ]:
plottings_logSpectral3(7,al7)

In [ ]:
plottings_spectrals3(7,al7)

In [ ]:
%matplotlib inline
unique,count = np.unique(predictions7,return_counts=True)
plt.stem(unique,count)
plt.title('{} ICs'.format(np.sum(count)))
plt.xlabel('clusters')
plt.ylabel('count')
plt.grid()

In [ ]:
order = [6,2,1,4,3,5,0]

In [ ]:
#### for 6 and 2
%matplotlib inline
ic_ = ic_comps.copy()
new_idx = np.delete(np.arange(n),np.r_[np.where(predictions7==6)[0],np.where(predictions7==2)[0]])
ic_[:,new_idx] = 0
cleaned_52 = np.dot(ic_, A.T) + mean
fig,ax = plt.subplots(cleaned_52.shape[1],1,figsize=(15,1.2*cleaned_52.shape[1]))
for i in range(cleaned_52.shape[1]):
    ax[i].plot(cleaned_52[:,i]+3,label=f'cleaned{i}',color='blue',lw=.5)
    ax[i].plot(traces[i,:],label=f'raw{i}',color='red',alpha=0.65,lw=.5)
    ax[i].vlines(x=[562,675,1103,1963,2162,2295,2296,2297,472,473,478,447,697,1020,2581,2600],ymin=traces[i,:].min(),ymax=traces[i,:].max(),ls='--',color='g',lw=.6)
    ax[i].legend()

In [ ]:
### for 6, 2, and 1
%matplotlib inline
ic_ = ic_comps.copy()
new_idx = np.delete(np.arange(n),np.r_[np.where(predictions7==6)[0],np.where(predictions7==2)[0],np.where(predictions7==1)[0]])
ic_[:,new_idx] = 0
cleaned_524 = np.dot(ic_, A.T) + mean
fig,ax = plt.subplots(cleaned_524.shape[1],1,figsize=(15,1.2*cleaned_524.shape[1]))
for i in range(cleaned_524.shape[1]):
    ax[i].plot(cleaned_524[:,i]+3,label=f'cleaned{i}',color='blue',lw=.5)
    ax[i].plot(traces[i,:],label=f'raw{i}',color='red',alpha=0.65,lw=.5)
    ax[i].vlines(x=[562,675,1103,1963,2162,2295,2296,2297,472,473,478,447,697,1020,2581,2600],ymin=traces[i,:].min(),ymax=traces[i,:].max(),ls='--',color='g',lw=.6)
    ax[i].legend()

In [ ]:
### for 6, 2, 1, and 4
%matplotlib inline
ic_ = ic_comps.copy()
new_idx = np.delete(np.arange(n),np.r_[np.where(predictions7==6)[0],np.where(predictions7==2)[0],np.where(predictions7==1)[0],np.where(predictions7==4)[0]])
ic_[:,new_idx] = 0
cleaned_6214 = np.dot(ic_, A.T) + mean
fig,ax = plt.subplots(cleaned_6214.shape[1],1,figsize=(15,1.2*cleaned_6214.shape[1]))
for i in range(cleaned_6214.shape[1]):
    ax[i].plot(cleaned_6214[:,i]+3,label=f'cleaned{i}',color='blue',lw=.5)
    ax[i].plot(traces[i,:],label=f'raw{i}',color='red',alpha=0.65,lw=.5)
    ax[i].vlines(x=[562,675,1103,1963,2162,2295,2296,2297,472,473,478,447,697,1020,2581,2600],ymin=traces[i,:].min(),ymax=traces[i,:].max(),ls='--',color='g',lw=.6)
    ax[i].legend()

In [ ]:
correlations = []
for i in range(traces.shape[0]):
    correlations.append(np.corrcoef(aa,traces[i,:])[1,0])
plt.figure(figsize=(18,5))
plt.stem(correlations)
plt.title('raw traces and behavior')
plt.ylim([-.4,.2])

In [ ]:
correlations_52 = []
for i in range(cleaned_52.shape[1]):
    correlations_52.append(np.corrcoef(aa,cleaned_52[:,i])[1,0])
plt.figure(figsize=(18,5))
plt.stem(correlations_52)
plt.title('2 clusters reprojected traces and behavior')
plt.ylim([-.4,.2])

In [ ]:
correlations_524 = []
for i in range(cleaned_524.shape[1]):
    correlations_524.append(np.corrcoef(aa,cleaned_524[:,i])[1,0])
plt.figure(figsize=(18,5))
plt.stem(correlations_524)
plt.title('3 clusters reprojected traces and behavior')
plt.ylim([-.4,.2])

### eig dec

In [ ]:
n = eig_dec(traces)

In [ ]:
a,b,c,d = ica_dec(traces,n,t=0.0001,max_=500)

In [ ]:
b.shape

In [ ]:
plott_ics(a)